# Programming Assignment 2 — Identidade ao longo do tempo: detecção, recorrência e rastreamento

**Disciplina:** Aprendizado Profundo — FGV EMAp  
**Professor:** Dario Oliveira | **Monitor:** Erick Brito  
**Integrantes:** Henrique Gabriel Gasparelo e [Nome do Colega]  

Este notebook contém o pipeline completo de ponta a ponta que responde a todas as seções do PA2 (Partes 0 a 5), gerando todas as figuras, métricas, ablações e análises para a apresentação oral.

## 0. Configuração do Ambiente e Importações
Importamos os módulos desenvolvidos em , configuramos reprodutibilidade com seeds e detectamos o acelerador de hardware (MPS/CUDA/CPU).

In [ ]:
import os
import sys
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Adicionar raiz e src ao path
for p in [".", "..", "src", "../src"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from src.utils import set_seed, get_device
from src.data import generate_synthetic_video, degrade_detections
from src.metrics import evaluate_tracking, run_unit_tests, compute_idf1, custom_nms
from src.tracker import Tracker
from src.models import VanillaRNNMotionModel, LSTMMotionModel, GRUMotionModel
from src.training import train_motion_model, run_ablation_axis1
from src.failures import compute_analytical_gradient_norm, compute_empirical_survival
from src.stress import run_detector_stress_test
from src.visualization import plot_decoupling_panel, plot_gradient_vanishing_curves

set_seed(42)
device = get_device()
print(f"Ambiente inicializado. Dispositivo: {device}")

---
## Parte 0 — Testes Sintéticos

Ambiente controlado onde conhecemos o ground truth exato.
1. **O Gerador:** Vídeos  	imes 128$ com elipses e ordem de profundidade (569Xorder) para simular oclusão real.
2. **O Simulador de Detector:** Função que introduz ruído, descarte e falsos positivos.
3. **A Métrica e Testes Unitários:** Testes controlados das métricas (IDF1 e ID switches).
4. **Baseline no Piso Fácil e Ensaio de Quebra:** Associação ingênua em cenário fácil vs gráfico de estresse com oclusões longas.

### 0.1 Gerador Sintético com Oclusão Real (569Xorder)

In [ ]:
# Gerar sequência sintética e visualizar trajetórias com oclusão verificável

### 0.2 Testes Unitários das Métricas (Casos a, b, c)
* **(a)** Predição = Ground Truth $\implies$ IDF1 = 1.0 e 0 switches;
* **(b)** Duas identidades trocadas a partir do quadro $ $\implies$ número exato de switches;
* **(c)** Uma track partida em duas no meio $\implies$ efeito esperado no IDF1.

In [ ]:
# Executar os 3 testes unitários formais das métricas
run_unit_tests()

### 0.3 Baseline no Piso Fácil e Gráfico de Quebra

In [ ]:
# Teste no piso fácil (sem oclusão) e curva de quebra aumentando velocidade e duração da oclusão
# [Espaço para execução da variação dos botões do gerador e plot do gráfico de quebra]

---
## Parte 1 — Baseline por Quadro

1. **Detecções:** Avaliação com detecções públicas do MOT17 (SDP escolhido como padrão justificadamente pela precisão superior) e detector pré-treinado torchvision Faster R-CNN (classe person).
2. **Associação Ingênua:** IoU entre quadros $ e -1$, Hungarian matching, threshold fixo, morte após $ quadros.
3. **Painel de Descolamento:** Gráfico obrigatório em dois painéis (mAP vs IDF1 e proporção de IDs vs switches).

In [ ]:
# Execução do baseline ingênuo sobre as sequências do MOT17
# [Espaço para carregar sequências e calcular as métricas IDF1, ID switches, fragmentações]

In [ ]:
# Renderização do Painel Obrigatório de Descolamento
# plot_decoupling_panel(metrics_por_sequencia)

---
## Parte 2 — Memória Temporal (Trilha A: RNN como Modelo de Movimento)

Substituição da associação ingênua por uma modelagem temporal de movimento:
* A rede recorrente recebe as caixas observadas 0$ e prevê a posição no quadro seguinte;
* Sob oclusão, o estado roda sem observação direta, propagando a previsão no escuro para manter a identidade;
* Perda Smooth L1 sobre as trajetórias ground truth.

In [ ]:
# Treinamento do modelo de movimento da Trilha A
# [Espaço para inicializar LSTMMotionModel e executar o treino com Smooth L1 Loss]

### Comparação Direta: Baseline Ingênuo vs Trilha A (Memória Temporal)
Apresentação das métricas lado a lado nas mesmas sequências de teste.

In [ ]:
# Tabela comparativa e ganhos de IDF1 e redução de ID switches

---
## Parte 3 — Ablação (Eixo 1: A Célula Recorrente e Janela de BPTT)

Estudo empírico comparando:
* **Células:** RNN Simples vs LSTM vs GRU (orçamento comparável de parâmetros);
* **Janela BPTT truncado:**  \in \{4, 8, 16, 32\}$;
* **Reprodutibilidade:** 3 seeds aleatórias com reporte de $	ext{média} \pm 	ext{desvio padrão}$.

In [ ]:
# Execução do grid de ablação multi-seed
# run_ablation_axis1(...)

---
## Parte 4 — Galeria de Falhas e Horizonte de Memória

1. **Medição Analítica:** Norma da derivada $\left\| \frac{\partial L_t}{\partial h_{t-k}} \right\|$ em função de $k$ (curva do gradiente que desaparece na RNN simples vs preservado nas portas de LSTM/GRU);
2. **Medição Empírica:** Tempo de sobrevivência do estado sob oclusão vs distribuição de oclusões do dataset;
3. **Galeria de 3 Falhas:** Diagnóstico das quebras estruturais;
4. **Correção:** Implementação de melhoria e avaliação do antes/depois.

In [ ]:
# Medição Analítica: Cálculo de ||dL_t / dh_{t-k}|| e plot comparativo
# plot_gradient_vanishing_curves(...)

In [ ]:
# Medição Empírica e Apresentação dos 3 Casos de Falha com Correção

---
## Parte 5 — Teste de Estresse (Degradação da Qualidade do Detector)

Aplicação do simulador de detector construído na Parte 0 sobre as detecções do MOT17 em 3 intensidades (leve, moderada, severa):
* Análise se o modelo temporal recorrente **absorve** ou **amplifica** as falhas do detector (reportando mAP e IDF1 juntos).

In [ ]:
# Execução do teste de estresse em 3 níveis
# run_detector_stress_test(...)

---
## Síntese dos Resultados e Roteiro da Apresentação Oral
Resumo executivo dos números finais, tabelas e gráficos gerados para a apresentação.